In [1]:
# PS 1:
# Adapt a pretrained model for a new classification task using feature extraction.

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
import numpy as np

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# subset for fast CPU execution
x_train = x_train[:2000]
y_train = y_train[:2000]

x_test = x_test[:500]
y_test = y_test[:500]

x_train = tf.image.resize(x_train, (96,96)) / 255.0
x_test = tf.image.resize(x_test, (96,96)) / 255.0

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96,96,3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(x_train, y_train, epochs=2)

loss, acc = model.evaluate(x_test, y_test)

print("Accuracy:", acc)

I0000 00:00:1778529492.246737  168707 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778529492.247097  168707 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778529492.399830  168707 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778529493.294398  168707 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 6s 1us/step
Epoch 1/2
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.5200 - loss: 1.4469
Epoch 2/2
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.7300 - loss: 0.8052
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.6700 - loss: 0.8839
Accuracy: 0.6700000166893005


In [2]:
# PS 2:
# Compare performance when using frozen layers versus partial fine-tuning.

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train = x_train[:2000]
y_train = y_train[:2000]

x_test = x_test[:500]
y_test = y_test[:500]

x_train = tf.image.resize(x_train, (96,96)) / 255.0
x_test = tf.image.resize(x_test, (96,96)) / 255.0

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96,96,3)
)

# partial fine tuning
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(x_train, y_train, epochs=2)

loss, acc = model.evaluate(x_test, y_test)

print("Accuracy:", acc)

Epoch 1/2
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.6195 - loss: 1.2417
Epoch 2/2
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.8515 - loss: 0.4545
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.3700 - loss: 3.7632
Accuracy: 0.3700000047683716


In [3]:
# PS 3:
# Analyse how pretrained features improve learning on small datasets.

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# very small dataset
x_train = x_train[:1000]
y_train = y_train[:1000]

x_test = x_test[:300]
y_test = y_test[:300]

x_train = tf.image.resize(x_train, (96,96)) / 255.0
x_test = tf.image.resize(x_test, (96,96)) / 255.0

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96,96,3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(x_train, y_train, epochs=2)

loss, acc = model.evaluate(x_test, y_test)

print("Accuracy with pretrained features:", acc)

Epoch 1/2
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.3360 - loss: 2.0203
Epoch 2/2
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6100 - loss: 1.0690
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6267 - loss: 1.0608
Accuracy with pretrained features: 0.6266666650772095


In [4]:
# PS 4:
# Evaluate training efficiency when only selected layers are updated.

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
import time

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train = x_train[:2000]
y_train = y_train[:2000]

x_test = x_test[:500]
y_test = y_test[:500]

x_train = tf.image.resize(x_train, (96,96)) / 255.0
x_test = tf.image.resize(x_test, (96,96)) / 255.0

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96,96,3)
)

for layer in base_model.layers[:-10]:
    layer.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

start = time.time()

model.fit(x_train, y_train, epochs=2)

end = time.time()

print("Training Time:", end - start, "seconds")

Epoch 1/2
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.6495 - loss: 1.1695
Epoch 2/2
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.8750 - loss: 0.3823
Training Time: 5.253357887268066 seconds
